<a href="https://colab.research.google.com/github/abiodunadesesan/FlyRank-ml-Internship-/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abiodunadesesan/FlyRank-ml-Internship-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [53]:
!pip -q install duckdb datasets huggingface_hub pyarrow pandas

In [54]:
from google.colab import userdata
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))

print("Successfully logged in to Hugging Face!")

Successfully logged in to Hugging Face!


In [55]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

print(dataset)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})


In [56]:
from datasets import load_dataset

daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True
)

print(daily)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

IterableDatasetDict({
    train: IterableDataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_shards: 18
    })
})


In [57]:
print(daily.keys())


dict_keys(['train'])


In [58]:
sample = next(iter(daily["train"]))
sample

{'report_date': datetime.date(2025, 1, 27),
 'client_hash_id': 'client_9958f0a7ae1df715',
 'content_hash_id': 'content_3b70a18ea133b2bb',
 'client_has_gsc': True,
 'client_has_ga4': True,
 'gsc_data_available': True,
 'ga4_data_available': False,
 'gsc_impressions': 30,
 'gsc_clicks': 0,
 'gsc_sum_position': 115,
 'gsc_avg_position': 3.8333333333333335,
 'ga4_pageviews': 0,
 'ga4_sessions': 0,
 'ga4_users': 0,
 'ga4_engaged_sessions': 0,
 'ga4_total_engagement_sec': 0,
 'sessions_organic': 0,
 'sessions_direct': 0,
 'sessions_referral': 0,
 'sessions_social': 0,
 'sessions_paid': 0,
 'sessions_ai': 0,
 'ai_chatgpt': 0,
 'ai_perplexity': 0,
 'ai_gemini': 0,
 'ai_copilot': 0,
 'ai_claude': 0,
 'ai_meta': 0,
 'ai_other': 0,
 'scroll_events': 0}

In [59]:
import duckdb
import pandas as pd

rows = []

for i, row in enumerate(daily["train"]):
    rows.append(row)
    if i >= 50000:
        break

df = pd.DataFrame(rows)

duckdb.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM df
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,duplicate_rows


In [60]:
duckdb.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM df
""").df()

,total_rows,start_date,end_date
0,50001,2025-01-27,2025-02-27


In [61]:
duckdb.sql("""
SELECT
    COUNT(*) AS available_rows
FROM df
WHERE
    gsc_data_available IS TRUE
    AND client_has_gsc IS TRUE
""").df()

,available_rows
0,50001


**Unit of analysis**

One row represents the daily search and analytics performance of one content item for one client on one report date.

**Table**
fact_content_daily_performance

**Time window**
March 2026 (month = "2026-03")

**Prediction**
Predict future Google Search clicks (`gsc_clicks`) using historical search and analytics metrics.

**Excluded**
Future information, label-derived columns, `client_hash_id`, `content_hash_id`, and rows where `gsc_data_available` or `ga4_data_available` are `FALSE`.

In [62]:
duckdb.sql("""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM df
""").df()

,start_date,end_date,total_rows
0,2025-01-27,2025-02-27,50001


### Feature
- gsc_impressions
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- sessions_organic

These are safe features because they are historical measurements available before making a prediction.

### Available when?

| Feature | Available when? |
|---------|------------------|
| gsc_impressions | Available because impressions are recorded before the prediction is made. |
| gsc_avg_position | Available because historical search position is known before the prediction date. |
| ga4_pageviews | Available because pageviews have already been observed before prediction. |
| ga4_sessions | Available because historical session counts exist before the decision moment. |
| sessions_organic | Available because organic sessions are measured before making the prediction. |

### Label
- gsc_clicks

This is the prediction target and must never be used as an input feature.

### Context
- report_date
- client_hash_id
- content_hash_id

These identify and group records but should not be used as predictive features.

### Excluded
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

These are used for filtering or describing data availability, not as prediction features.

In [63]:
feature_df = df[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions",
        "sessions_organic",
        "gsc_clicks"
    ]
].copy()

feature_df.head()

,gsc_impressions,gsc_avg_position,ga4_pageviews,ga4_sessions,sessions_organic,gsc_clicks
0,30,3.833333,0,0,0,0
1,5,71.600000,0,0,0,0
2,1,34.000000,0,0,0,0
3,6,23.333333,0,0,0,0
4,5,17.800000,0,0,0,0


In [64]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Fill missing values with 0 for this demonstration
clean_df = feature_df.fillna(0)

X = clean_df.drop(columns=["gsc_clicks"])
y = clean_df["gsc_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

model = LinearRegression()
model.fit(X_train, y_train)

print("Honest R²:", model.score(X_test, y_test))

Honest R²: 0.30153283957250376


In [65]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Intentionally leak the label by adding it back as a feature
leak_df = clean_df.copy()
leak_df["leaked_clicks"] = leak_df["gsc_clicks"]

X = leak_df.drop(columns=["gsc_clicks"])
y = leak_df["gsc_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

print("Leaked R²:", model.score(X_test, y_test))

Leaked R²: 1.0


## Verification Summary

The contract was verified using three queries:

1. Grain check: No duplicate combinations of `report_date`, `client_hash_id`, and `content_hash_id` were found, confirming that each row represents one content item for one client on one report date.

2. Row count and date span: The sample contained **50,001 rows**, covering **2025-01-27** to **2025-02-27**.

3. Availability check: Filtering with `gsc_data_available IS TRUE` and `client_has_gsc IS TRUE` retained **50,001 rows**, confirming that the selected records contain valid Google Search Console data.

In [66]:
# Verify row count and date range

print("Total rows:", len(df))
print("Start date:", df["report_date"].min())
print("End date:", df["report_date"].max())

print("\nDuplicate rows:")
print(
    df.duplicated(
        subset=["report_date", "client_hash_id", "content_hash_id"]
    ).sum()
)

print("\nRows with GSC available:")
print(df["gsc_data_available"].sum())

Total rows: 50001
Start date: 2025-01-27
End date: 2025-02-27

Duplicate rows:
0

Rows with GSC available:
50001


## Data limits

This analysis uses only a subset of the warehouse data rather than the full dataset.

History length differs between clients, so results from this sample may not represent all clients equally.

Rows before GA4 data collection may contain zero-filled analytics values when `ga4_data_available` is `FALSE`, so those values should not be interpreted as actual user activity.

The leakage experiment intentionally included a label-derived feature to demonstrate why future or target information must never be used as a model feature. The honest model result (R² ≈ 0.30) is the valid performance measure.

In [67]:
missing = df[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions",
        "sessions_organic",
        "gsc_clicks"
    ]
].isnull().sum()

print(missing)

gsc_impressions     0
gsc_avg_position    1
ga4_pageviews       0
ga4_sessions        0
sessions_organic    0
gsc_clicks          0
dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.